In [ ]:
# Code-mixing Script
# ---------------------
# Reads a CSV with columns: Title, Category, Body
# Extracts topic-relevant keywords from each sentence using embedding similarity
# Translates keywords into German
# Capitalizes only German nouns/proper nouns with a German POS tagger
# Replaces selected English keywords in the original text with German translations
# Repetition handling applied only to translated keywords actually used in the sentence
# Writes output as .jsonl, one JSON object per line

import re
import json
import pandas as pd
import numpy as np
import torch
from collections import OrderedDict
from sentence_transformers import SentenceTransformer
from transformers import MarianMTModel, MarianTokenizer
import spacy
from spacy.cli import download
from sklearn.feature_extraction.text import CountVectorizer


## Configuration

INPUT_CSV = "../data/scidcc/SciDCC.csv"
OUTPUT_JSONL = "SciDCC-codeswitched.jsonl"

TITLE_COL = "Title"
CATEGORY_COL = "Category"
TEXT_COL = "Body"

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TRANSLATE_MODEL_NAME = "Helsinki-NLP/opus-mt-en-de"

NGRAM_RANGE = (2, 3)  # keyword length range during extraction
TOP_K_PER_SENTENCE = 4  # maximum number of keywords replaced in each sentence
SIM_THRESHOLD = 0.30  # minimum embedding similarity score required for a candidate keyword to be considered relevant; higher values = stricter selection, fewer keywords; lower values = more keywords, but potentially more noise.

CHUNK_WORDS = 180  # prevents losing content from long articles due to model input limits
CHUNK_OVERLAP = 20  # number of overlapping words between consecutive chunks to preserve context

BATCH_SIZE_EMBED = 128  # number of texts embedded at once by the sentence transformer (larger values are faster on GPU, but use more VRAM)
BATCH_SIZE_TRANSLATE = 64  # number of phrases translated at once by the translation model (larger values improve speed but require more GPU memory)
PRINT_EVERY = 10  # print progress every n processed rows

# Repetition handling only for translated keywords
MAX_TRANSLATED_REPETITIONS_PER_SENTENCE = 1
MAX_SOURCE_KEYWORD_REPETITIONS_PER_SENTENCE = 1

MAX_CONSECUTIVE_USED_TRANSLATION_PHRASE_COPIES = 1 # if a translated phrase that was actually used in a sentence is repeated, keep at most this many copies



device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## Models

embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)

translator_tokenizer = MarianTokenizer.from_pretrained(TRANSLATE_MODEL_NAME)
translator_model = MarianMTModel.from_pretrained(TRANSLATE_MODEL_NAME).to(device)
translator_model.eval()

# English sentence splitter
nlp_en = spacy.blank("en")
nlp_en.add_pipe("sentencizer")

# German POS tagger for capitalisation
try:
    nlp_de = spacy.load("de_core_news_sm")
except Exception:
    download("de_core_news_sm")
    nlp_de = spacy.load("de_core_news_sm")



def remove_dot_noise(text: str) -> str:
    text = str(text)
    text = re.sub(r'(?:\s*\.\s*){4,}', ' ', text)
    text = re.sub(r'\.{4,}', ' ', text)
    text = re.sub(r'(?:\s*[-_]\s*){6,}', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalise_dashes(text: str) -> str:
    text = str(text)
    return text.replace("–", "—").replace("--", "—")

def normalise_spaces(text: str) -> str:
    text = normalise_dashes(text)
    text = remove_dot_noise(text)
    return re.sub(r"\s+", " ", str(text)).strip()

def split_into_sentences(text: str):
    text = normalise_spaces(text)
    if not text:
        return []

    doc = nlp_en(text)
    sents = [remove_dot_noise(s.text.strip()) for s in doc.sents if s.text.strip()]
    sents = [s for s in sents if s]
    return sents if sents else [text]

def repair_glued_words(text: str) -> str:
    text = str(text)
    text = re.sub(r'(?<=[a-zäöüß])(?=[A-ZÄÖÜ])', ' ', text)
    text = re.sub(r'\b([a-zäöüß]{4,})([A-ZÄÖÜ][a-zäöüß]{3,})\b', r'\1 \2', text)
    return re.sub(r'\s{2,}', ' ', text).strip()

def postprocess_codeswitched_text(text: str) -> str:
    text = str(text)
    text = normalise_dashes(text)
    text = remove_dot_noise(text)
    text = repair_glued_words(text)
    text = re.sub(r'\s+([,;:.!?])', r'\1', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

def chunk_text_by_words(text: str, chunk_words=180, overlap=30):
    words = text.split()
    if not words:
        return []
    if len(words) <= chunk_words:
        return [text]

    chunks = []
    step = max(1, chunk_words - overlap)
    for start in range(0, len(words), step):
        end = min(len(words), start + chunk_words)
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
    return chunks

def mean_pool_embeddings(texts, batch_size=BATCH_SIZE_EMBED):
    if not texts:
        dim = embedder.get_sentence_embedding_dimension()
        return np.zeros(dim, dtype=np.float32)

    embs = embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embs.mean(axis=0)

def article_topic_embedding(article_text: str):
    chunks = chunk_text_by_words(article_text, CHUNK_WORDS, CHUNK_OVERLAP)
    return mean_pool_embeddings(chunks)

def clean_candidate(c: str) -> str:
    c = c.strip(" -–—,;:.!?\"'()[]{}")
    c = re.sub(r"\s+", " ", c)
    return c.strip()

def generate_candidates(sentence: str, ngram_range=(1, 3)):
    sentence = normalise_spaces(sentence)
    if not sentence:
        return []

    vectorizer = CountVectorizer(
        ngram_range=ngram_range,
        stop_words="english",
        lowercase=False,
        token_pattern=r"(?u)\b[A-Za-z][A-Za-z\-]+\b"
    )

    try:
        vectorizer.fit([sentence])
        candidates = vectorizer.get_feature_names_out().tolist()
    except ValueError:
        return []

    filtered = []
    seen = set()

    for c in candidates:
        c2 = clean_candidate(c)
        if not c2:
            continue
        if len(c2) < 3:
            continue
        if c2.lower() in seen:
            continue
        if re.fullmatch(r"[-–—]+", c2):
            continue
        seen.add(c2.lower())
        filtered.append(c2)

    return filtered

def cosine_sim_matrix_to_vector(matrix, vector):
    return np.dot(matrix, vector)


# German capitalisation
def capitalize_only_german_nouns(phrase: str) -> str:
    if not phrase or not phrase.strip():
        return phrase

    doc = nlp_de(phrase)
    out = []

    for tok in doc:
        txt = tok.text
        if tok.is_punct:
            out.append(txt + tok.whitespace_)
        elif txt.isupper() and len(txt) > 1:
            out.append(txt + tok.whitespace_)
        elif tok.pos_ in {"NOUN", "PROPN"}:
            out.append((txt[:1].upper() + txt[1:]) + tok.whitespace_ if txt else tok.whitespace_)
        else:
            out.append(txt.lower() + tok.whitespace_)

    return "".join(out).strip()

# translation clean-up (only the translated phrase itself)
def clean_translation_phrase(phrase: str) -> str:
    phrase = normalise_spaces(phrase)
    
    # (EZB/EZB/EZB) -> (EZB)
    phrase = re.sub(
        r'\((?P<tok>[A-Za-zÀ-ÖØ-öø-ÿ0-9.\-]+)(?:/\s*(?P=tok))+\)',
        r'(\g<tok>)',
        phrase,
        flags=re.IGNORECASE
    )

    # EZB/EZB/EZB -> EZB
    phrase = re.sub(
        r'(?<!\w)(?P<tok>[A-Za-zÀ-ÖØ-öø-ÿ0-9.\-]+)(?:/\s*(?P=tok))+(?!\w)',
        r'\g<tok>',
        phrase,
        flags=re.IGNORECASE
    )

    # (foo)(foo)(foo) -> (foo)
    phrase = re.sub(r'(\([^()]{1,200}\))(?:\s*\1)+', r'\1', phrase)

    # single-letter repeats inside the translation: s s s s -> s
    phrase = re.sub(r'(?<!\w)(?P<tok>[A-Za-z])(?:\s+(?P=tok)){2,}(?!\w)', r'\g<tok>', phrase)

    phrase = re.sub(r'\s{2,}', ' ', phrase).strip()
    return phrase

def normalised_phrase_key(s: str) -> str:
    s = normalise_spaces(s).lower()
    s = re.sub(r'[“”„"]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def phrase_tokens(s: str):
    return re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", normalised_phrase_key(s))

def is_bad_information_translation(de: str) -> bool: # skip super short translations like 's'.
    toks = phrase_tokens(de)
    if not toks:
        return True
    if len(toks) == 1 and len(toks[0]) <= 1:
        return True
    return False

# skip a translation if it is already covered by a longer translation in the same sentence
def translation_is_redundant(new_de: str, accepted_des) -> bool:
    new_key = normalised_phrase_key(new_de)
    new_toks = phrase_tokens(new_de)

    if not new_key:
        return True

    for used in accepted_des:
        used_key = normalised_phrase_key(used)
        used_toks = phrase_tokens(used)

        # if exact same translation
        if new_key == used_key:
            return True

        # if shorter phrase contained inside longer phrase
        if len(new_key) < len(used_key):
            if re.search(rf'(?<!\w){re.escape(new_key)}(?!\w)', used_key, flags=re.IGNORECASE):
                return True

        if new_toks and used_toks and len(new_toks) < len(used_toks):
            if set(new_toks).issubset(set(used_toks)):
                return True

    return False


## Repetition handling for used translations only

# collapse exact repeated translated phrases only if they were used as translations in the sentence
def collapse_used_translation_phrase_repetitions(sentence: str, used_translations, max_keep=1):
    sentence = str(sentence)
    if not used_translations:
        return sentence

    phrases = sorted(
        {p.strip() for p in used_translations if p and p.strip()},
        key=lambda x: (-len(x.split()), -len(x))
    )

    for phrase in phrases:
        esc = re.escape(phrase)
        sep = r'(?:\s*,\s*|\s*;\s*|\s*:\s*|\s*—\s*|\s*-\s*|\s+)'
        pattern = re.compile(
            rf'(?<!\w)(?P<phrase>{esc})(?P<tail>(?:{sep}(?P=phrase)){{{max_keep},}})(?!\w)',
            flags=re.IGNORECASE
        )

        prev = None
        while prev != sentence:
            prev = sentence

            def repl(m):
                matched = m.group(0)
                phrase_text = m.group("phrase")

                if "—" in matched:
                    joiner = " — "
                elif re.search(r'\s-\s|-', matched):
                    joiner = " - "
                elif "," in matched:
                    joiner = ", "
                elif ";" in matched:
                    joiner = "; "
                elif ":" in matched:
                    joiner = ": "
                else:
                    joiner = " "

                return joiner.join([phrase_text] * max_keep)

            sentence = pattern.sub(repl, sentence)

    return re.sub(r'\s{2,}', ' ', sentence).strip()

# (Ozon Recovery) (Ozon Recovery) -> (Ozon Recovery)
# (EZB/EZB/EZB) -> (EZB)   only if EZB is a used translation
def collapse_used_translation_parenthetical_repetitions(sentence: str, used_translations):
    sentence = str(sentence)
    if not sentence or not used_translations:
        return sentence

    phrases = sorted(
        {p.strip() for p in used_translations if p and p.strip()},
        key=lambda x: (-len(x.split()), -len(x))
    )

    # repeated "(phrase)" copies
    for phrase in phrases:
        esc = re.escape(phrase)
        pattern = re.compile(
            rf'(?:\(\s*{esc}\s*\))(?:\s*(?:\(\s*{esc}\s*\)))+',
            flags=re.IGNORECASE
        )
        sentence = pattern.sub(f'({phrase})', sentence)

    # slash-chains EZB/EZB/EZB
    used_single_tokens = {
        normalised_phrase_key(p)
        for p in phrases
        if len(phrase_tokens(p)) == 1
    }

    if used_single_tokens:
        for tok in sorted(used_single_tokens, key=len, reverse=True):
            tok_esc = re.escape(tok)

            # (EZB/EZB/EZB) -> (EZB)
            sentence = re.sub(
                rf'\((?P<tok>{tok_esc})(?:/\s*(?P=tok))+\)',
                lambda m: f'({m.group("tok")})',
                sentence,
                flags=re.IGNORECASE
            )

            # EZB/EZB/EZB -> EZB
            sentence = re.sub(
                rf'(?<!\w)(?P<tok>{tok_esc})(?:/\s*(?P=tok))+(?!\w)',
                lambda m: m.group("tok"),
                sentence,
                flags=re.IGNORECASE
            )

    # remove duplicated parentheses
    sentence = re.sub(r'(\([^()]{1,200}\))(?:\s*\1)+', r'\1', sentence)
    return re.sub(r'\s{2,}', ' ', sentence).strip()

# s s s s s -> s
# EZB EZB EZB -> EZB
# Kohlenstoff Kohlenstoff -> Kohlenstoff  (only for used translated tokens)
def collapse_used_translation_single_token_repetitions(sentence: str, used_translations, max_keep=1):
    sentence = str(sentence)
    if not used_translations:
        return sentence

    used_single_tokens = []
    for p in used_translations:
        toks = phrase_tokens(p)
        if len(toks) == 1:
            used_single_tokens.append(toks[0])

    if not used_single_tokens:
        return sentence

    for tok in sorted(set(used_single_tokens), key=len, reverse=True):
        tok_esc = re.escape(tok)
        sep = r'(?:\s*,\s*|\s*;\s*|\s*:\s*|\s+)'

        pattern = re.compile(
            rf'(?<!\w)(?P<tok>{tok_esc})(?P<tail>(?:{sep}(?P=tok)){{{max_keep},}})(?!\w)',
            flags=re.IGNORECASE
        )

        prev = None
        while prev != sentence:
            prev = sentence

            def repl(m):
                matched = m.group(0)
                tok_text = m.group("tok")
                if "," in matched:
                    joiner = ", "
                elif ";" in matched:
                    joiner = "; "
                elif ":" in matched:
                    joiner = ": "
                else:
                    joiner = " "
                return joiner.join([tok_text] * max_keep)

            sentence = pattern.sub(repl, sentence)

    return re.sub(r'\s{2,}', ' ', sentence).strip()


# Caches
_translation_cache = {}
_embedding_cache = {}


## Translation

def translate_phrases_to_german(phrases):
    if not phrases:
        return []

    uncached = [p for p in phrases if p not in _translation_cache]

    if uncached:
        for i in range(0, len(uncached), BATCH_SIZE_TRANSLATE):
            batch = uncached[i:i+BATCH_SIZE_TRANSLATE]

            inputs = translator_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)

            with torch.no_grad():
                generated = translator_model.generate(
                    **inputs,
                    max_length=128,
                    num_beams=4
                )

            decoded = translator_tokenizer.batch_decode(generated, skip_special_tokens=True)

            for src, de in zip(batch, decoded):
                de = normalise_spaces(de)
                de = capitalize_only_german_nouns(de)
                de = clean_translation_phrase(de)
                _translation_cache[src] = de

    return [_translation_cache[p] for p in phrases]


def encode_texts_with_cache(texts):
    if not texts:
        return np.empty((0, embedder.get_sentence_embedding_dimension()), dtype=np.float32)

    unique_texts = []
    for t in texts:
        if t not in _embedding_cache:
            unique_texts.append(t)

    if unique_texts:
        embs = embedder.encode(
            unique_texts,
            batch_size=BATCH_SIZE_EMBED,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        for t, e in zip(unique_texts, embs):
            _embedding_cache[t] = e

    return np.stack([_embedding_cache[t] for t in texts], axis=0)


## Translations Replacement

def find_all_non_overlapping_matches(sentence, phrase):
    pattern = re.compile(rf"(?<!\w){re.escape(phrase)}(?!\w)", flags=re.IGNORECASE)
    return list(pattern.finditer(sentence))

def find_non_overlapping_replacements(sentence, candidates, scores, top_k=3, threshold=0.28):
    items = [(cand, score) for cand, score in zip(candidates, scores) if score >= threshold]
    items.sort(key=lambda x: (x[1], len(x[0])), reverse=True)

    selected = []
    occupied = []

    for cand, score in items:
        matches = find_all_non_overlapping_matches(sentence, cand)
        if not matches:
            continue

        for match in matches:
            s, e = match.span()
            overlap = any(not (e <= os or s >= oe) for os, oe in occupied)
            if overlap:
                continue

            selected.append((cand, s, e, score))
            occupied.append((s, e))

            if len(selected) >= top_k:
                break

        if len(selected) >= top_k:
            break

    return selected

# repetition only for selected translated keywords; suppress shorter redundant translations covered by longer ones
def should_apply_translation(src, de, source_counts, translated_counts, accepted_translations):
    if not de or not de.strip():
        return False

    if is_bad_information_translation(de):
        return False

    src_key = src.lower().strip()
    de_key = normalised_phrase_key(de)

    if source_counts.get(src_key, 0) >= MAX_SOURCE_KEYWORD_REPETITIONS_PER_SENTENCE:
        return False

    if translated_counts.get(de_key, 0) >= MAX_TRANSLATED_REPETITIONS_PER_SENTENCE:
        return False

    if translation_is_redundant(de, accepted_translations):
        return False

    return True


## Process article

def prepare_sentence_candidates(sentences):
    sent_candidates = []
    all_candidates = []

    for sent in sentences:
        cands = generate_candidates(sent, ngram_range=NGRAM_RANGE)
        sent_candidates.append(cands)
        all_candidates.extend(cands)

    unique_candidates = list(dict.fromkeys(all_candidates))
    return sent_candidates, unique_candidates

def codemix_article(body_text: str):
    body_text = normalise_spaces(body_text)
    if not body_text:
        return body_text

    sentences = split_into_sentences(body_text)
    if not sentences:
        return body_text

    art_emb = article_topic_embedding(body_text)

    sent_candidates, unique_candidates = prepare_sentence_candidates(sentences)

    if unique_candidates:
        unique_cand_embs = encode_texts_with_cache(unique_candidates)
        cand2emb = {c: e for c, e in zip(unique_candidates, unique_cand_embs)}
    else:
        cand2emb = {}

    sentence_replacements = []
    all_selected_phrases = []

    for sent, candidates in zip(sentences, sent_candidates):
        if not candidates:
            sentence_replacements.append([])
            continue

        cand_embs = np.stack([cand2emb[c] for c in candidates], axis=0)
        sims = cosine_sim_matrix_to_vector(cand_embs, art_emb)

        selected = find_non_overlapping_replacements(
            sent,
            candidates,
            sims,
            top_k=TOP_K_PER_SENTENCE,
            threshold=SIM_THRESHOLD
        )

        sentence_replacements.append(selected)
        all_selected_phrases.extend([cand for cand, _, _, _ in selected])

    unique_selected = list(dict.fromkeys(all_selected_phrases))
    if unique_selected:
        translated_unique = translate_phrases_to_german(unique_selected)
        translation_map = dict(zip(unique_selected, translated_unique))
    else:
        translation_map = {}

    new_sentences = []
    for sent, selected in zip(sentences, sentence_replacements):
        if not selected:
            new_sentences.append(sent)
            continue

        # decide which selected replacements to apply
        decision_items = []
        for src, start, end, score in selected:
            de = translation_map.get(src, src)
            decision_items.append({
                "src": src,
                "start": start,
                "end": end,
                "score": score,
                "de": de
            })

        decision_items.sort(
            key=lambda x: (-x["score"], -len(x["src"]), -len(x["de"]))
        )

        source_counts = {}
        translated_counts = {}
        accepted_translations = []
        accepted_items = []

        for item in decision_items:
            src = item["src"]
            de = item["de"]

            if not should_apply_translation(
                src, de,
                source_counts=source_counts,
                translated_counts=translated_counts,
                accepted_translations=accepted_translations
            ):
                continue

            accepted_items.append(item)
            accepted_translations.append(de)

            src_key = src.lower().strip()
            de_key = normalised_phrase_key(de)
            source_counts[src_key] = source_counts.get(src_key, 0) + 1
            translated_counts[de_key] = translated_counts.get(de_key, 0) + 1

        accepted_items.sort(key=lambda x: x["start"], reverse=True)

        new_sent = sent
        used_translations_in_sentence = []

        for item in accepted_items:
            new_sent = new_sent[:item["start"]] + item["de"] + new_sent[item["end"]:]
            used_translations_in_sentence.append(item["de"])

        # repetition cleanup only for translations used in this sentence
        new_sent = collapse_used_translation_phrase_repetitions(
            new_sent,
            used_translations=used_translations_in_sentence,
            max_keep=MAX_CONSECUTIVE_USED_TRANSLATION_PHRASE_COPIES
        )

        new_sent = collapse_used_translation_parenthetical_repetitions(
            new_sent,
            used_translations=used_translations_in_sentence
        )

        new_sent = collapse_used_translation_single_token_repetitions(
            new_sent,
            used_translations=used_translations_in_sentence,
            max_keep=1
        )

        new_sentences.append(new_sent)

    final_text = " ".join(new_sentences).strip()
    final_text = postprocess_codeswitched_text(final_text)
    return final_text


## Main

df = pd.read_csv(INPUT_CSV)

required_cols = {TITLE_COL, CATEGORY_COL, TEXT_COL}

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f_out:
    for idx, row in df.iterrows():
        title = "" if pd.isna(row[TITLE_COL]) else str(row[TITLE_COL])
        category = "" if pd.isna(row[CATEGORY_COL]) else str(row[CATEGORY_COL])
        body = "" if pd.isna(row[TEXT_COL]) else str(row[TEXT_COL])

        try:
            codemixed = codemix_article(body)
        except Exception as e:
            print(f"Row {idx} failed, error: {e}")
            codemixed = body

        obj = OrderedDict({
            "Title": title,
            "Category": category,
            "codeswitched": codemixed
        })

        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f_out.flush()

        if (idx + 1) % PRINT_EVERY == 0:
            print(f"Processed {idx + 1}/{len(df)} rows")

print(f"Done. Output written to: {OUTPUT_JSONL}")